# L03 · 刚体物理与稳定仿真

本实验将从接触现象走向数值证据：通过受控的时间离散对照，用多项信号测量接触，再检验接触双方的材料如何共同影响滑动。所有场景只使用内置 Box，因此不需要下载资产。数值核心路径不依赖渲染；在具备渲染能力的机器上，还可以启用可选的 Genesis 相机路径，在定量曲线之外查看真实的初始与最终场景画面。

## 运行前准备

GPU 是本课程的首选后端。保持默认的 `ROBO_GENESIS_BACKEND=auto` 即可；在经过验证的 AMD ROCm 环境中，每个 runner 都会选择 `gs.amdgpu`，并报告实际使用的后端。

只有在没有可用的已验证 GPU 时，才把 CPU 作为速度较慢的兼容路径。仅当你明确需要这一 fallback，或希望复现 CPU 参考路径时，才设置 `ROBO_GENESIS_BACKEND=cpu`。Genesis 初始化具有进程级状态，因此每个 case 都在独立进程中运行，本 notebook 的 kernel 不会调用 `gs.init()`。

渲染是与仿真后端分开的显式能力选择。保持 `ROBO_GENESIS_RENDER=0` 会运行纯状态路径，打印渲染 `SKIP`，并使用根据实测状态绘制的示意图。在启动 kernel 前设置 `ROBO_GENESIS_RENDER=1`，会在 `build()` 前加入离屏相机并采集真实 Genesis 画面；一旦启用，相机或渲染错误就会使运行失败，不会静默退回示意图。

生成的 `.npz`、相机画面和图片会写入 `ROBO_GENESIS_OUTPUTS_DIR` 指定的目录；未设置时写入仓库的 `outputs/`。runner 会同时报告请求后端和实际后端。如果 AMD 运行已经开始却发生失败，错误会保持可见，不会被 CPU 结果替代。

In [ ]:
import importlib.metadata as package_metadata
import os
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from robo_genesis.course_utils import notebook_mode


def installed_version(distribution):
    try:
        return package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        return "not installed"


def load_case(path):
    with np.load(path, allow_pickle=False) as archive:
        return {name: archive[name].copy() for name in archive.files}


def scalar(case, name):
    return case[name].item()


def validate_render_frames(case, label):
    assert bool(scalar(case, "render_enabled")) == render_enabled, label
    for field in ("initial_rgb", "final_rgb"):
        image = case[field]
        if render_enabled:
            assert image.ndim == 3 and image.shape[0] > 0 and image.shape[1] > 0, (label, field, image.shape)
            assert image.shape[2] in (3, 4), (label, field, image.shape)
            assert np.isfinite(image).all(), (label, field)
        else:
            assert image.shape == (0,), (label, field, image.shape)


def run_process(command, label):
    print("$ " + shlex.join(command), flush=True)
    completed = subprocess.run(command, text=True, capture_output=True, check=False)
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="", file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"{label} failed with return code {completed.returncode}")
    return completed


backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")

render_value = os.environ.get("ROBO_GENESIS_RENDER", "0").strip()
if render_value not in {"0", "1"}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == "1"

runtime = notebook_mode("l03-rigid-body-physics", show_viewer=False)
output_dir = runtime["output_dir"]
contact_dir = (output_dir / "contact").resolve()
friction_dir = (output_dir / "friction").resolve()
contact_dir.mkdir(parents=True, exist_ok=True)
friction_dir.mkdir(parents=True, exist_ok=True)

environment = {
    "python": sys.version.split()[0],
    "genesis_world": installed_version("genesis-world"),
    "torch": installed_version("torch"),
    "requested_backend": backend_mode,
    "render_enabled": render_enabled,
    "output_dir": str(output_dir.resolve()),
}
for key, value in environment.items():
    print(f"{key:>20}: {value}")

## Part A · 仿真前先预测

四个 case 使用完全相同的方块、桌面、密度、摩擦、初始位姿、seed、精度和 1.5 秒仿真时长，只改变 `dt` 和 `substeps`。运行后续单元前，先写下你的答案：

1. 哪些 case 会产生 150 个外层样本，哪些只有 75 个？
2. 固定 `dt` 增加 `substeps` 时，你预计结果会怎样变化？
3. N1 和 N4 的 `substep_dt` 相同。它们在共同采样时刻的状态是否应当接近？完整数组长度是否也应相同？
4. 如果穿透减小但出现短暂分离，能否只凭其中一个数字认定该 case 更稳定？

In [ ]:
# Physical inputs are written here so you can inspect the experiment directly.
TABLE_CENTER_Z = 0.70
TABLE_HEIGHT = 0.05
TABLE_SIZE = (0.90, 0.60, TABLE_HEIGHT)
TABLE_POSITION = (0.35, 0.0, TABLE_CENTER_Z)
TABLE_FRICTION = 0.80
CUBE_SIZE = 0.08
CUBE_INITIAL_POSITION = (0.35, 0.0, 1.0)
CUBE_DENSITY = 500.0
CUBE_FRICTION = 0.50
EXPECTED_CENTER_Z = TABLE_CENTER_Z + TABLE_HEIGHT / 2 + CUBE_SIZE / 2
SIM_DURATION = 1.5


def validated_step_count(duration, dt):
    if duration <= 0 or dt <= 0:
        raise ValueError("duration and dt must be positive")
    n_steps = round(duration / dt)
    if n_steps < 1 or not np.isclose(n_steps * dt, duration, rtol=0.0, atol=1e-12):
        raise ValueError("duration must be an integer multiple of dt")
    return n_steps

CASE_SPECS = [
    {"label": "N1", "dt": 0.01, "substeps": 1},
    {"label": "N2", "dt": 0.01, "substeps": 2},
    {"label": "N3", "dt": 0.02, "substeps": 1},
    {"label": "N4", "dt": 0.02, "substeps": 2},
]
CONTACT_SCENE = {
    "table_size_m": TABLE_SIZE,
    "table_position_m": TABLE_POSITION,
    "table_friction": TABLE_FRICTION,
    "cube_size_m": (CUBE_SIZE,) * 3,
    "cube_initial_position_m": CUBE_INITIAL_POSITION,
    "cube_density_kg_m3": CUBE_DENSITY,
    "cube_friction": CUBE_FRICTION,
    "expected_resting_center_z_m": EXPECTED_CENTER_Z,
}
print("Fixed Part A scene:")
for name, value in CONTACT_SCENE.items():
    print(f"  {name}: {value}")

configuration_rows = []
for spec in CASE_SPECS:
    n_steps = validated_step_count(SIM_DURATION, spec["dt"])
    simulated_time = n_steps * spec["dt"]
    substep_dt = spec["dt"] / spec["substeps"]
    internal_updates = n_steps * spec["substeps"]
    assert np.isclose(simulated_time, SIM_DURATION)
    configuration_rows.append(
        {**spec, "n_steps": n_steps, "simulated_time": simulated_time,
         "substep_dt": substep_dt, "internal_updates": internal_updates}
    )

assert configuration_rows[0]["n_steps"] == configuration_rows[1]["n_steps"] == 150
assert configuration_rows[2]["n_steps"] == configuration_rows[3]["n_steps"] == 75
assert np.isclose(configuration_rows[0]["substep_dt"], configuration_rows[3]["substep_dt"])
assert configuration_rows[0]["n_steps"] != configuration_rows[3]["n_steps"]

lines = [
    "| Case | dt [s] | substeps | substep_dt [s] | samples | internal updates | duration [s] |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for row in configuration_rows:
    lines.append(
        "| {label} | {dt:.3f} | {substeps} | {substep_dt:.4f} | {n_steps} | "
        "{internal_updates} | {simulated_time:.1f} |".format(**row)
    )
display(Markdown("\n".join(lines)))

### 阅读一组隔离接触实验实际执行的代码

`robo_genesis.experiments.rigid_contact` 只是执行边界，并不隐藏实验逻辑。每个子进程都会在其 `main()` 函数中执行下面这段完全相同的代码；其中的常量就是上方已经打印的物理输入。参数解析、派生指标计算、`.npz` 序列化和终端摘要仍保留在可复用模块中，这里不重复展示。

```python
n_steps = validated_step_count(args.duration, args.dt)

import genesis as gs
import torch

if args.backend == "cpu":
    backend = gs.cpu
    print("Backend: CPU (explicit request)")
else:
    backend = select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision="32", logging_level="warning")
if getattr(gs, "amdgpu", None) is not None and gs.backend == gs.amdgpu:
    actual_backend = "amdgpu"
elif gs.backend == gs.cpu:
    actual_backend = "cpu"
else:
    actual_backend = str(gs.backend)
genesis_version = package_metadata.version("genesis-world")
torch_version = str(torch.__version__)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=args.dt, substeps=args.substeps),
    rigid_options=gs.options.RigidOptions(enable_collision=True),
    show_viewer=False,
)
scene.add_entity(gs.morphs.Plane())
table = scene.add_entity(
    gs.morphs.Box(size=TABLE_SIZE, pos=TABLE_POSITION, fixed=True),
    material=gs.materials.Rigid(friction=TABLE_FRICTION),
    surface=gs.surfaces.Default(color=(0.55, 0.38, 0.22, 1.0)),
)
cube = scene.add_entity(
    gs.morphs.Box(size=(CUBE_SIZE,) * 3, pos=CUBE_INITIAL_POSITION),
    material=gs.materials.Rigid(rho=CUBE_DENSITY, friction=CUBE_FRICTION),
    surface=gs.surfaces.Default(color=(0.15, 0.70, 0.45, 1.0)),
)
camera = None
if args.render:
    camera = scene.add_camera(
        res=(640, 360),
        pos=(1.45, -1.35, 1.25),
        lookat=(0.35, 0.0, 0.78),
        fov=42,
        GUI=False,
    )
scene.build()
initial_cube_pos = to_numpy(cube.get_pos()).reshape(-1).copy()

def render_rgb(label: str) -> np.ndarray:
    if camera is None:
        return np.empty((0,), dtype=np.uint8)
    rgb = to_numpy(camera.render(rgb=True)[0])
    if rgb.ndim != 3 or rgb.shape[0] == 0 or rgb.shape[1] == 0 or rgb.shape[2] not in (3, 4):
        raise AssertionError(f"{label}: expected a non-empty HxWx3/4 image, got {rgb.shape}")
    if not np.isfinite(rgb).all():
        raise AssertionError(f"{label}: image contains non-finite values")
    return rgb.copy()

initial_rgb = render_rgb("initial_rgb")

# Record state once after each outer scene.step(). Event times are therefore
# quantized by dt rather than resolved continuously at substep_dt.
time_values = np.arange(1, n_steps + 1, dtype=float) * args.dt
z_history = np.empty(n_steps, dtype=float)
vz_history = np.empty(n_steps, dtype=float)
contact_counts = np.empty(n_steps, dtype=int)

for index in range(n_steps):
    scene.step()
    z_history[index] = float(to_numpy(cube.get_pos()).reshape(-1)[2])
    vz_history[index] = float(to_numpy(cube.get_vel()).reshape(-1)[2])
    contacts = cube.get_contacts(with_entity=table)
    contact_counts[index] = int(contacts["position"].shape[0])
final_rgb = render_rgb("final_rgb")
```

运行前先沿着数据流阅读：`dt` 和 `substeps` 进入 `SimOptions`；桌面和方块携带固定的物理输入；每次外层 `scene.step()` 产生一个高度、竖直速度和接触数量样本。启用渲染时，相机会在 `build()` 前加入，并采集真实的初始帧和最终帧。runner 会保存这些原始数组和可选画面，后面的分析单元会重新计算派生指标，并与保存值逐项核对。

由于 Genesis 初始化是进程级状态，实验仍必须使用子进程。把这段代码展示在这里，既能让你直接阅读实验，也不需要在 notebook kernel 中初始化 Genesis 或复制另一份可执行实现。

### 在隔离进程中运行四组接触实验

打印出的命令会显式展示每个发生变化的参数，包括可选的 `--render` 标志。每个 runner 会把配置、后端与版本信息、采样状态、接触证据、派生指标，以及两帧经过校验的相机画面或明确的空渲染字段保存到一个 `.npz` 文件。如果子进程失败，notebook 会先显示完整 stdout 和 stderr，再停止执行。

In [ ]:
CONTACT_FIELDS = {
    "requested_backend", "actual_backend", "genesis_version", "torch_version", "torch_hip",
    "render_enabled", "initial_rgb", "final_rgb",
    "seed", "precision", "dt", "substeps", "substep_dt", "duration",
    "n_steps", "table_size", "table_position", "table_friction",
    "cube_size", "cube_initial_position", "cube_density", "cube_friction", "time",
    "z", "vz", "contact_count", "expected_center_z", "first_contact_time",
    "penetration", "zero_contact_duration", "real_separation_duration",
    "max_rebound_clearance", "max_upward_vz", "contact_presence_transitions",
    "contact_count_changes", "settling_error", "initial_cube_pos", "final_cube_pos",
}

contact_results = {}
contact_commands = {}
for spec in CASE_SPECS:
    label = spec["label"]
    output_path = contact_dir / f"{label.lower()}.npz"
    command = [
        sys.executable, "-m", "robo_genesis.experiments.rigid_contact",
        "--backend", backend_mode,
        "--dt", str(spec["dt"]),
        "--substeps", str(spec["substeps"]),
        "--duration", str(SIM_DURATION),
        "--output", str(output_path),
    ]
    if render_enabled:
        command.append("--render")
    contact_commands[label] = command
    run_process(command, f"contact case {label}")
    case = load_case(output_path)
    missing = CONTACT_FIELDS - set(case)
    if missing:
        raise KeyError(f"contact case {label} is missing fields: {sorted(missing)}")
    validate_render_frames(case, f"contact case {label}")
    contact_results[label] = case

print("PASS — all four contact cases completed in isolated processes")
if render_enabled:
    print("PASS — Genesis camera captured initial and final frames for N1–N4")
else:
    print("SKIP — rendering disabled; the next cell will use measured-state schematics")

### 先读指标定义，再查看结果

runner 在每次外层 `scene.step()` 后采样一次，因此事件时间分辨率是外层 `dt`，而不是内部时间步；短于 `dt` 的事件可能不会被观测到。

`Case`、`backend`、`dt`、`substeps`、`internal dt` 和 `samples` 都是上文已经介绍的直接配置或运行字段。下表只解释派生或容易误读的诊断量。

| 诊断输出列 | 本实验中的定义 | 解读方式与边界 |
|---|---|---|
| `penetration proxy [mm]` | `max(0, 理想静止中心高度 − 最低采样中心高度)` | 它是基于采样中心高度的代理值，不是求解器精确的连续时间穿透。 |
| `geometric separation [ms]` | 首次接触后，无报告接触且方块底部间隙大于 `1e-5 m` 的样本数乘以 `dt` | 比只看接触数更能支持分离判断，但仍受 `dt` 量化。 |
| `max upward vz [m/s]` | 首次接触后最大的正向竖直速度 | 表示向上运动；还要结合几何分离和高度轨迹，才能判断是否发生反弹。 |
| `contact-count changes` | 首次接触后，相邻样本所报接触数不同的次数 | 表示接触流形的变化，不是可单独使用的稳定性分数，而且会受采样节奏影响。 |
| `settling error [mm]` | 理想静止高度与最后 0.2 s 平均中心高度的绝对差 | 越小表示尾段均值越接近理想高度，但均值仍可能掩盖振荡。 |

建议先比较 `internal dt` 与穿透代理值，再检查几何分离和高度轨迹，最后把接触数变化作为辅助证据。没有任何单独一列能完整判断稳定性。

In [ ]:
def first_true_index(mask):
    indices = np.flatnonzero(mask)
    return int(indices[0]) if indices.size else -1


def contact_metrics_from_samples(case):
    dt = scalar(case, "dt")
    z = case["z"]
    vz = case["vz"]
    contact_count = case["contact_count"]
    expected_center_z = scalar(case, "expected_center_z")
    contact_mask = contact_count > 0
    first_index = first_true_index(contact_mask)
    first_time = case["time"][first_index] if first_index >= 0 else np.nan
    penetration = max(0.0, expected_center_z - float(np.min(z)))

    if first_index >= 0:
        post_contact = contact_mask[first_index:]
        post_counts = contact_count[first_index:]
        clearance = z[first_index:] - CUBE_SIZE / 2 - (TABLE_CENTER_Z + TABLE_HEIGHT / 2)
        separation = (~post_contact) & (clearance > 1e-5)
        zero_contact_duration = float(np.count_nonzero(~post_contact) * dt)
        real_separation_duration = float(np.count_nonzero(separation) * dt)
        max_rebound_clearance = max(0.0, float(np.max(clearance)))
        max_upward_vz = max(0.0, float(np.max(vz[first_index:])))
        contact_presence_transitions = int(np.count_nonzero(np.diff(post_contact.astype(int))))
        contact_count_changes = int(np.count_nonzero(np.diff(post_counts)))
    else:
        zero_contact_duration = np.nan
        real_separation_duration = np.nan
        max_rebound_clearance = np.nan
        max_upward_vz = np.nan
        contact_presence_transitions = 0
        contact_count_changes = 0

    tail_samples = max(1, round(0.2 / dt))
    settling_error = abs(float(np.mean(z[-tail_samples:])) - expected_center_z)
    return {
        "first_contact_time": first_time,
        "penetration": penetration,
        "zero_contact_duration": zero_contact_duration,
        "real_separation_duration": real_separation_duration,
        "max_rebound_clearance": max_rebound_clearance,
        "max_upward_vz": max_upward_vz,
        "contact_presence_transitions": contact_presence_transitions,
        "contact_count_changes": contact_count_changes,
        "settling_error": settling_error,
    }


contact_rows = []
contact_data_checks = {}
contact_metrics = {}
for spec in CASE_SPECS:
    label = spec["label"]
    case = contact_results[label]
    expected_samples = round(SIM_DURATION / spec["dt"])
    arrays_ok = all(
        case[name].shape == (expected_samples,) and np.isfinite(case[name]).all()
        for name in ("time", "z", "vz", "contact_count")
    )
    metadata_ok = (
        scalar(case, "requested_backend") == backend_mode
        and scalar(case, "actual_backend") in {"cpu", "amdgpu"}
        and np.isclose(scalar(case, "dt"), spec["dt"])
        and int(scalar(case, "substeps")) == spec["substeps"]
        and int(scalar(case, "n_steps")) == expected_samples
        and np.isclose(scalar(case, "duration"), SIM_DURATION)
        and np.allclose(case["table_size"], TABLE_SIZE)
        and np.allclose(case["table_position"], TABLE_POSITION)
        and np.isclose(scalar(case, "table_friction"), TABLE_FRICTION)
        and np.isclose(scalar(case, "cube_size"), CUBE_SIZE)
        and np.allclose(case["cube_initial_position"], CUBE_INITIAL_POSITION)
        and np.isclose(scalar(case, "cube_density"), CUBE_DENSITY)
        and np.isclose(scalar(case, "cube_friction"), CUBE_FRICTION)
    )
    metrics = contact_metrics_from_samples(case)
    contact_metrics[label] = metrics
    for name, recomputed in metrics.items():
        saved = scalar(case, name)
        if np.isnan(recomputed):
            assert np.isnan(saved), name
        else:
            assert np.isclose(saved, recomputed), (name, saved, recomputed)
    contact_observed = np.isfinite(metrics["first_contact_time"])
    not_through_table = float(np.min(case["z"])) > TABLE_CENTER_Z
    contact_data_checks[label] = arrays_ok and metadata_ok and contact_observed and not_through_table
    contact_rows.append({
        "label": label,
        "backend": scalar(case, "actual_backend"),
        "dt": scalar(case, "dt"),
        "substeps": int(scalar(case, "substeps")),
        "substep_dt": scalar(case, "substep_dt"),
        "samples": len(case["time"]),
        "penetration_mm": 1000 * metrics["penetration"],
        "separation_ms": 1000 * metrics["real_separation_duration"],
        "max_upward_vz": metrics["max_upward_vz"],
        "contact_changes": int(metrics["contact_count_changes"]),
        "settling_error_mm": 1000 * metrics["settling_error"],
    })

lines = [
    "| Case | backend | dt [s] | substeps | internal dt [s] | samples | penetration proxy [mm] | geometric separation [ms] | max upward vz [m/s] | contact-count changes | settling error [mm] |",
    "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for row in contact_rows:
    lines.append(
        "| {label} | {backend} | {dt:.3f} | {substeps} | {substep_dt:.4f} | {samples} | "
        "{penetration_mm:.3f} | {separation_ms:.1f} | {max_upward_vz:.4f} | "
        "{contact_changes} | {settling_error_mm:.4f} |".format(**row)
    )
display(Markdown("\n".join(lines)))
assert all(contact_data_checks.values()), contact_data_checks

In [ ]:
CONTACT_COLORS = {"N1": "#247BA0", "N2": "#2A9D5B", "N3": "#D95F43", "N4": "#7555A6"}


def draw_contact_state(axis, cube_position, title, cube_color):
    axis.add_patch(plt.Rectangle(
        (0.35 - 0.9 / 2, TABLE_CENTER_Z - TABLE_HEIGHT / 2),
        0.9, TABLE_HEIGHT, color="#8C6138", label="fixed table",
    ))
    axis.add_patch(plt.Rectangle(
        (cube_position[0] - CUBE_SIZE / 2, cube_position[2] - CUBE_SIZE / 2),
        CUBE_SIZE, CUBE_SIZE, color=cube_color, label="dynamic cube",
    ))
    axis.set(xlim=(-0.18, 0.88), ylim=(0.62, 1.08), xlabel="x [m]", ylabel="z [m]")
    axis.set_title(title)
    axis.set_aspect("equal")
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)


n1 = contact_results["N1"]
state_figure, state_axes = plt.subplots(1, len(CASE_SPECS) + 1, figsize=(18, 3.5))
if render_enabled:
    state_axes[0].imshow(n1["initial_rgb"])
    state_axes[0].axis("off")
else:
    draw_contact_state(state_axes[0], n1["initial_cube_pos"], "", "#2A9D8F")
state_axes[0].set_title("Shared initial scene")
for axis, spec in zip(state_axes[1:], CASE_SPECS):
    label = spec["label"]
    case = contact_results[label]
    if render_enabled:
        axis.imshow(case["final_rgb"])
        axis.axis("off")
    else:
        draw_contact_state(axis, case["final_cube_pos"], "", CONTACT_COLORS[label])
    axis.set_title(f"{label}: dt={spec['dt']}, sub={spec['substeps']}")
visual_source = "Genesis camera frames" if render_enabled else "Measured-state schematics — rendering disabled"
state_figure.suptitle(f"{visual_source}: same scene and duration")
state_figure.tight_layout()
contact_state_path = output_dir / "contact_initial_final.png"
state_figure.savefig(contact_state_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(state_figure)

trajectory_figure, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
for spec in CASE_SPECS:
    label = spec["label"]
    case = contact_results[label]
    legend = f"{label}: dt={spec['dt']}, substeps={spec['substeps']}"
    axes[0].plot(case["time"], 1000 * (case["z"] - scalar(case, "expected_center_z")), color=CONTACT_COLORS[label], label=legend)
    axes[1].plot(case["time"], case["vz"], color=CONTACT_COLORS[label], label=legend)
    axes[2].step(case["time"], case["contact_count"], where="post", color=CONTACT_COLORS[label], label=legend)
axes[0].axhline(0.0, color="#444444", linestyle="--", linewidth=1)
axes[0].set(ylabel="center-height error [mm]", title="Height relative to ideal rest")
axes[1].axhline(0.0, color="#444444", linewidth=1)
axes[1].set(ylabel="vertical velocity [m/s]", title="Vertical motion")
axes[2].set(xlabel="simulated time [s]", ylabel="contact count", title="Sampled contact evidence")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
trajectory_figure.tight_layout()
contact_trajectory_path = output_dir / "contact_trajectories.png"
trajectory_figure.savefig(contact_trajectory_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(trajectory_figure)
print("saved:", contact_state_path.resolve())
print("saved:", contact_trajectory_path.resolve())

In [ ]:
def shared_sample_indices(left_time, right_time, decimals=12):
    left = np.asarray(left_time, dtype=float)
    right = np.asarray(right_time, dtype=float)
    if left.ndim != 1 or right.ndim != 1 or left.size == 0 or right.size == 0:
        raise ValueError("timelines must be non-empty 1-D arrays")
    if np.any(np.diff(left) <= 0) or np.any(np.diff(right) <= 0):
        raise ValueError("timelines must be strictly increasing")
    _, left_indices, right_indices = np.intersect1d(
        np.round(left, decimals), np.round(right, decimals), return_indices=True
    )
    if left_indices.size == 0:
        raise ValueError("timelines have no common sample timestamps")
    return left_indices, right_indices


n1, n2, n3, n4 = (contact_results[label] for label in ("N1", "N2", "N3", "N4"))
n1_indices, n4_indices = shared_sample_indices(n1["time"], n4["time"])
shared_times = n1["time"][n1_indices]
assert np.allclose(shared_times, n4["time"][n4_indices])
shared_z_difference = float(np.max(np.abs(n1["z"][n1_indices] - n4["z"][n4_indices])))
shared_vz_difference = float(np.max(np.abs(n1["vz"][n1_indices] - n4["vz"][n4_indices])))
SHARED_Z_TOLERANCE = 1e-5
SHARED_VZ_TOLERANCE = 1e-4
penetrations = {label: contact_metrics[label]["penetration"] for label in contact_results}
part_a_checks = {
    **{f"{label}_data_valid": passed for label, passed in contact_data_checks.items()},
    "N1_to_N2_penetration_decreased": penetrations["N2"] < penetrations["N1"],
    "N3_to_N4_penetration_decreased": penetrations["N4"] < penetrations["N3"],
    "N1_N4_shared_z_within_tolerance": shared_z_difference <= SHARED_Z_TOLERANCE,
    "N1_N4_shared_vz_within_tolerance": shared_vz_difference <= SHARED_VZ_TOLERANCE,
}

fixed_dt_lines = []
for left_label, right_label in (("N1", "N2"), ("N3", "N4")):
    left_case = contact_results[left_label]
    right_case = contact_results[right_label]
    direction = (
        "decreased"
        if penetrations[right_label] < penetrations[left_label]
        else "did not decrease"
    )
    fixed_dt_lines.append(
        f"- {left_label} → {right_label} keeps outer dt at "
        f"{1000 * scalar(left_case, 'dt'):.1f} ms while the internal step changes from "
        f"{1000 * scalar(left_case, 'substep_dt'):.1f} to "
        f"{1000 * scalar(right_case, 'substep_dt'):.1f} ms. The penetration proxy "
        f"{direction}: {1000 * penetrations[left_label]:.3f} → "
        f"{1000 * penetrations[right_label]:.3f} mm."
    )

internal_steps_match = np.isclose(scalar(n1, "substep_dt"), scalar(n4, "substep_dt"))
shared_samples_within_tolerance = (
    part_a_checks["N1_N4_shared_z_within_tolerance"]
    and part_a_checks["N1_N4_shared_vz_within_tolerance"]
)
shared_sample_verdict = "within" if shared_samples_within_tolerance else "outside"

separation_lines = []
for label in ("N1", "N2", "N3", "N4"):
    metrics = contact_metrics[label]
    separation_ms = 1000 * metrics["real_separation_duration"]
    zero_contact_ms = 1000 * metrics["zero_contact_duration"]
    clearance_mm = 1000 * metrics["max_rebound_clearance"]
    upward_vz = metrics["max_upward_vz"]
    if separation_ms > 0:
        verdict = (
            f"geometric separation was observed for {separation_ms:.1f} ms; "
            f"maximum clearance was {clearance_mm:.3f} mm and maximum upward vz was "
            f"{upward_vz:.4f} m/s"
        )
    elif zero_contact_ms > 0:
        verdict = (
            f"zero-contact samples totaled {zero_contact_ms:.1f} ms, but positive "
            f"geometric separation was not observed; maximum upward vz was "
            f"{upward_vz:.4f} m/s"
        )
    else:
        verdict = (
            "no post-contact zero-contact interval or geometric separation was observed; "
            f"maximum upward vz was {upward_vz:.4f} m/s"
        )
    separation_lines.append(
        f"- {label}: {verdict}. Contact count changed "
        f"{metrics['contact_count_changes']} times and tail settling error was "
        f"{1000 * metrics['settling_error']:.3f} mm."
    )

guided_interpretation = (
    "### Guided interpretation\n\n"
    "**1. Hold `dt` fixed and change `substeps`.**\n\n"
    + "\n".join(fixed_dt_lines)
    + "\n\n**2. Match `substep_dt` and change the external step boundary.**\n\n"
    + f"N1 and N4 internal steps {'match' if internal_steps_match else 'do not match'} at "
    + f"{1000 * scalar(n1, 'substep_dt'):.1f} and "
    + f"{1000 * scalar(n4, 'substep_dt'):.1f} ms. Their outer steps are "
    + f"{1000 * scalar(n1, 'dt'):.1f} and {1000 * scalar(n4, 'dt'):.1f} ms, so they "
    + f"produce {len(n1['time'])} and {len(n4['time'])} samples. Across "
    + f"{len(shared_times)} shared timestamps, maximum differences are "
    + f"{1000 * shared_z_difference:.6f} mm in z and "
    + f"{shared_vz_difference:.6g} m/s in vz, {shared_sample_verdict} the declared "
    + f"tolerances. First observed contact occurs at "
    + f"{contact_metrics['N1']['first_contact_time']:.3f} s for N1 and "
    + f"{contact_metrics['N4']['first_contact_time']:.3f} s for N4; this timestamp is "
    + "quantized by each outer dt and is not, by itself, evidence of different collision physics.\n\n"
    + "**3. Scope the conclusion.**\n\n"
    + f"These measurements come from Genesis {scalar(n1, 'genesis_version')} on the "
    + f"{scalar(n1, 'actual_backend')} backend, with this geometry, seed, duration, and "
    + "parameter range. They support the comparisons above; they do not prove that dt "
    + "never matters or that one setting is universally stable. Commands, sensors, and "
    + "callbacks update at outer `scene.step()` boundaries, so changing dt also changes "
    + "their cadence. A smaller penetration proxy remains only one piece of evidence.\n\n"
    + "**4. Do not infer rebound from contact count alone.**\n\n"
    + "\n".join(separation_lines)
    + "\n\nUse geometric clearance and vertical velocity together with contact count, "
    + "settling error, and the full trajectories before describing rebound or stability."
)
display(Markdown(guided_interpretation))
print(part_a_checks)

## Part B · 摩擦与持续停止

两个相同的自由方块先在同一张桌面上沉降，再获得相同的 `2.0 m/s` 水平速度。基准实验只改变方块摩擦。Genesis 1.3.3 在默认运行时 ratio 下取两个刚体摩擦值中的较大值。运行前请先预测：

1. 桌面摩擦为 0.50，两个方块摩擦分别为 0.10 和 0.80 时，两个接触对有效摩擦系数是多少？
2. 哪条通道在持续停止前应当移动得更远？
3. 为什么 runner 除了记录 `x(t)` 和 `vx(t)`，还必须记录角速度？
4. 只交换橙色和蓝色 Surface，会改变两条轨迹吗？

### 阅读隔离摩擦实验实际执行的代码

`robo_genesis.experiments.rigid_friction` 使用与 Part A 相同的执行边界。基准实验和修改桌面后的每个子进程都会在其 `main()` 函数中执行下面这段完全相同的代码；下一个单元负责提供并打印物理与数值输入。参数解析、持续停止指标计算、`.npz` 序列化和终端摘要仍保留在可复用模块中，这里不重复展示。

```python
settle_steps = validated_step_count(args.settle_duration, args.dt, label="settle-duration")
measure_steps = validated_step_count(args.measure_duration, args.dt, label="measure-duration")
hold_samples = max(1, round(args.stop_hold / args.dt))

import genesis as gs
import torch

if args.backend == "cpu":
    backend = gs.cpu
    print("Backend: CPU (explicit request)")
else:
    backend = select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision="32", logging_level="warning")
if getattr(gs, "amdgpu", None) is not None and gs.backend == gs.amdgpu:
    actual_backend = "amdgpu"
elif gs.backend == gs.cpu:
    actual_backend = "cpu"
else:
    actual_backend = str(gs.backend)
genesis_version = package_metadata.version("genesis-world")
torch_version = str(torch.__version__)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=args.dt, substeps=args.substeps),
    rigid_options=gs.options.RigidOptions(enable_collision=True),
    show_viewer=False,
)
table = scene.add_entity(
    gs.morphs.Box(size=TABLE_SIZE, pos=TABLE_POSITION, fixed=True),
    material=gs.materials.Rigid(friction=args.table_friction),
    surface=gs.surfaces.Default(color=(0.55, 0.38, 0.22, 1.0)),
)
cube_low = scene.add_entity(
    gs.morphs.Box(size=(CUBE_SIZE,) * 3, pos=(START_X, LOW_LANE_Y, RESTING_CENTER_Z)),
    material=gs.materials.Rigid(rho=CUBE_DENSITY, friction=args.low_friction),
    surface=gs.surfaces.Default(color=(0.95, 0.50, 0.15, 1.0)),
)
cube_high = scene.add_entity(
    gs.morphs.Box(size=(CUBE_SIZE,) * 3, pos=(START_X, HIGH_LANE_Y, RESTING_CENTER_Z)),
    material=gs.materials.Rigid(rho=CUBE_DENSITY, friction=args.high_friction),
    surface=gs.surfaces.Default(color=(0.15, 0.45, 0.85, 1.0)),
)
camera = None
if args.render:
    camera = scene.add_camera(
        res=(720, 400),
        pos=(1.25, -1.55, 1.35),
        lookat=(0.0, 0.0, 0.72),
        fov=46,
        GUI=False,
    )
scene.build()

# Establish the contact manifold before defining the horizontal-motion t=0.
for _ in range(settle_steps):
    scene.step()

start_low = to_numpy(cube_low.get_pos()).reshape(-1).copy()
start_high = to_numpy(cube_high.get_pos()).reshape(-1).copy()

def render_rgb(label: str) -> np.ndarray:
    if camera is None:
        return np.empty((0,), dtype=np.uint8)
    rgb = to_numpy(camera.render(rgb=True)[0])
    if rgb.ndim != 3 or rgb.shape[0] == 0 or rgb.shape[1] == 0 or rgb.shape[2] not in (3, 4):
        raise AssertionError(f"{label}: expected a non-empty HxWx3/4 image, got {rgb.shape}")
    if not np.isfinite(rgb).all():
        raise AssertionError(f"{label}: image contains non-finite values")
    return rgb.copy()

initial_rgb = render_rgb("initial_rgb")
initial_dofs_velocity = np.array([args.initial_vx, 0.0, 0.0, 0.0, 0.0, 0.0])
cube_low.set_dofs_velocity(initial_dofs_velocity)
cube_high.set_dofs_velocity(initial_dofs_velocity)

time_values = np.arange(measure_steps + 1, dtype=float) * args.dt
x_low = np.empty(measure_steps + 1, dtype=float)
x_high = np.empty(measure_steps + 1, dtype=float)
vx_low = np.empty(measure_steps + 1, dtype=float)
vx_high = np.empty(measure_steps + 1, dtype=float)
omega_y_low = np.empty(measure_steps + 1, dtype=float)
omega_y_high = np.empty(measure_steps + 1, dtype=float)
contacts_low = np.empty(measure_steps + 1, dtype=int)
contacts_high = np.empty(measure_steps + 1, dtype=int)

def record(index):
    x_low[index] = float(to_numpy(cube_low.get_pos()).reshape(-1)[0])
    x_high[index] = float(to_numpy(cube_high.get_pos()).reshape(-1)[0])
    vx_low[index] = float(to_numpy(cube_low.get_vel()).reshape(-1)[0])
    vx_high[index] = float(to_numpy(cube_high.get_vel()).reshape(-1)[0])
    omega_y_low[index] = float(to_numpy(cube_low.get_ang()).reshape(-1)[1])
    omega_y_high[index] = float(to_numpy(cube_high.get_ang()).reshape(-1)[1])
    contacts_low[index] = int(cube_low.get_contacts(with_entity=table)["position"].shape[0])
    contacts_high[index] = int(cube_high.get_contacts(with_entity=table)["position"].shape[0])

record(0)
for index in range(1, measure_steps + 1):
    scene.step()
    record(index)
final_rgb = render_rgb("final_rgb")
```

可以把实验读成三个阶段：先让两个方块沉降并建立桌面接触，并在启用渲染时采集一帧真实画面；再给它们设置相同的六自由度速度向量；最后在每个外层 step 后同步采样平移、转动和接触证据，并采集最终画面。runner 保存这些原始轨迹和可选画面，后续分析单元再重新计算持续停止时刻和距离，并与保存值核对。

Surface 颜色只用于区分两条通道。真正的物理对照来自 `args.low_friction` 与 `args.high_friction`，而桌面摩擦同时参与两个接触对。

In [ ]:
# Part B keeps geometry and density fixed while changing material friction.
FRICTION_TABLE_SIZE = (2.0, 0.8, TABLE_HEIGHT)
FRICTION_TABLE_POSITION = (0.0, 0.0, TABLE_CENTER_Z)
FRICTION_CUBE_SIZE = 0.08
FRICTION_CUBE_DENSITY = 500.0
RESTING_CENTER_Z = TABLE_CENTER_Z + TABLE_HEIGHT / 2 + FRICTION_CUBE_SIZE / 2
START_X = -0.60
LOW_LANE_Y = -0.15
HIGH_LANE_Y = 0.15


def effective_pair_friction(table_friction, cube_friction):
    if table_friction < 0 or cube_friction < 0:
        raise ValueError("friction values must be non-negative")
    return max(table_friction, cube_friction)


FRICTION_SPEC = {
    "dt": 0.01,
    "substeps": 2,
    "settle_duration": 0.30,
    "measure_duration": 2.00,
    "initial_vx": 2.00,
    "table_friction": 0.50,
    "low_friction": 0.10,
    "high_friction": 0.80,
    "stop_speed": 0.01,
    "stop_hold": 0.10,
}
FRICTION_SCENE = {
    "table_size_m": FRICTION_TABLE_SIZE,
    "table_position_m": FRICTION_TABLE_POSITION,
    "cube_size_m": (FRICTION_CUBE_SIZE,) * 3,
    "cube_density_kg_m3": FRICTION_CUBE_DENSITY,
    "start_x_m": START_X,
    "lane_y_m": (LOW_LANE_Y, HIGH_LANE_Y),
    "resting_center_z_m": RESTING_CENTER_Z,
}
print("Fixed Part B scene:")
for name, value in FRICTION_SCENE.items():
    print(f"  {name}: {value}")
print("Part B baseline inputs:")
for name, value in FRICTION_SPEC.items():
    print(f"  {name}: {value}")


def run_friction_case(spec, output_name):
    output_path = friction_dir / output_name
    command = [
        sys.executable, "-m", "robo_genesis.experiments.rigid_friction",
        "--backend", backend_mode,
        "--dt", str(spec["dt"]),
        "--substeps", str(spec["substeps"]),
        "--settle-duration", str(spec["settle_duration"]),
        "--measure-duration", str(spec["measure_duration"]),
        "--initial-vx", str(spec["initial_vx"]),
        "--table-friction", str(spec["table_friction"]),
        "--low-friction", str(spec["low_friction"]),
        "--high-friction", str(spec["high_friction"]),
        "--stop-speed", str(spec["stop_speed"]),
        "--stop-hold", str(spec["stop_hold"]),
        "--output", str(output_path),
    ]
    if render_enabled:
        command.append("--render")
    run_process(command, f"friction case {output_name}")
    return load_case(output_path)


baseline_friction = run_friction_case(FRICTION_SPEC, "baseline.npz")
validate_render_frames(baseline_friction, "baseline friction case")
print("effective low lane:", effective_pair_friction(0.50, 0.10))
print("effective high lane:", effective_pair_friction(0.50, 0.80))

In [ ]:
def sustained_stop_index(speed, threshold, hold_samples):
    speed = np.asarray(speed, dtype=float)
    if speed.ndim != 1 or speed.size == 0 or not np.isfinite(speed).all():
        raise ValueError("speed must be a non-empty, finite 1-D array")
    if threshold < 0 or hold_samples < 1:
        raise ValueError("threshold must be non-negative and hold_samples positive")
    below_threshold = np.abs(speed) < threshold
    for index in range(len(speed) - hold_samples + 1):
        if np.all(below_threshold[index:index + hold_samples]):
            return index
    return -1


def stop_measurement(case, suffix):
    hold_samples = int(scalar(case, "hold_samples"))
    stop_index = sustained_stop_index(
        case[f"vx_{suffix}"], scalar(case, "stop_speed"), hold_samples
    )
    if stop_index < 0:
        return {"index": -1, "time": np.nan, "distance": np.nan}
    return {
        "index": stop_index,
        "time": float(case["time"][stop_index]),
        "distance": float(case[f"x_{suffix}"][stop_index] - case[f"x_{suffix}"][0]),
    }


FRICTION_FIELDS = {
    "requested_backend", "actual_backend", "genesis_version", "torch_version", "torch_hip",
    "render_enabled", "initial_rgb", "final_rgb",
    "seed", "precision", "dt", "substeps", "substep_dt",
    "settle_duration", "measure_duration", "settle_steps", "measure_steps",
    "initial_vx", "table_friction", "low_friction", "high_friction",
    "effective_low_friction", "effective_high_friction", "stop_speed",
    "stop_hold", "hold_samples", "table_size", "table_position",
    "cube_size", "cube_density", "resting_center_z", "low_lane_y",
    "high_lane_y", "start_x", "start_low", "start_high", "final_low", "final_high",
    "stop_time_low", "stop_time_high", "stop_distance_low", "stop_distance_high",
}
missing_friction_fields = FRICTION_FIELDS - set(baseline_friction)
if missing_friction_fields:
    raise KeyError(f"baseline friction result is missing fields: {sorted(missing_friction_fields)}")
FRICTION_ARRAY_FIELDS = (
    "time", "x_low", "x_high", "vx_low", "vx_high",
    "omega_y_low", "omega_y_high", "contacts_low", "contacts_high",
)
expected_measure_samples = round(FRICTION_SPEC["measure_duration"] / FRICTION_SPEC["dt"]) + 1
for name in FRICTION_ARRAY_FIELDS:
    values = baseline_friction[name]
    assert values.shape == (expected_measure_samples,), (name, values.shape)
    assert np.isfinite(values).all(), name
baseline_stop_measurements = {
    suffix: stop_measurement(baseline_friction, suffix) for suffix in ("low", "high")
}
for suffix, measurement in baseline_stop_measurements.items():
    for field in ("time", "distance"):
        saved = scalar(baseline_friction, f"stop_{field}_{suffix}")
        recomputed = measurement[field]
        if np.isnan(recomputed):
            assert np.isnan(saved), (suffix, field)
        else:
            assert np.isclose(saved, recomputed), (suffix, field, saved, recomputed)
assert scalar(baseline_friction, "requested_backend") == backend_mode
assert scalar(baseline_friction, "actual_backend") in {"cpu", "amdgpu"}
for name, expected in FRICTION_SPEC.items():
    assert np.isclose(scalar(baseline_friction, name), expected), name
assert np.isclose(scalar(baseline_friction, "effective_low_friction"), 0.50)
assert np.isclose(scalar(baseline_friction, "effective_high_friction"), 0.80)

def format_measurement(value, digits=4):
    return f"{value:.{digits}f}" if np.isfinite(value) else "not stopped in window"

def format_distance(value):
    return f"{value:.4f} m" if np.isfinite(value) else "not stopped in window"

friction_rows = []
for label, suffix in (("Low-μ cube", "low"), ("High-μ cube", "high")):
    friction_rows.append({
        "label": label,
        "cube": scalar(baseline_friction, f"{suffix}_friction"),
        "table": scalar(baseline_friction, "table_friction"),
        "effective": scalar(baseline_friction, f"effective_{suffix}_friction"),
        "stop_time": baseline_stop_measurements[suffix]["time"],
        "stop_distance": baseline_stop_measurements[suffix]["distance"],
    })
lines = [
    "| Lane | cube μ | table μ | effective pair μ | sustained stop [s] | stop distance [m] |",
    "|---|---:|---:|---:|---:|---:|",
]
for row in friction_rows:
    lines.append(
        f"| {row['label']} | {row['cube']:.2f} | {row['table']:.2f} | "
        f"{row['effective']:.2f} | {format_measurement(row['stop_time'], 3)} | "
        f"{format_measurement(row['stop_distance'])} |"
    )
display(Markdown("\n".join(lines)))

def draw_friction_state(axis, low_position, high_position, title):
    axis.add_patch(plt.Rectangle((-1.0, -0.4), 2.0, 0.8, color="#8C6138"))
    for position, color, label in (
        (low_position, "#F27A24", "low μ"),
        (high_position, "#2674C8", "high μ"),
    ):
        axis.add_patch(plt.Rectangle(
            (position[0] - 0.04, position[1] - 0.04), 0.08, 0.08, color=color, label=label
        ))
    axis.set(xlim=(-1.05, 1.05), ylim=(-0.45, 0.45), xlabel="x [m]", ylabel="y [m]")
    axis.set_aspect("equal")
    axis.set_title(title)
    axis.legend(fontsize=8)

friction_figure, friction_axes = plt.subplots(2, 2, figsize=(12, 8))
if render_enabled:
    friction_axes[0, 0].imshow(baseline_friction["initial_rgb"])
    friction_axes[0, 1].imshow(baseline_friction["final_rgb"])
    for axis in friction_axes[0]:
        axis.axis("off")
else:
    draw_friction_state(friction_axes[0, 0], baseline_friction["start_low"], baseline_friction["start_high"], "")
    draw_friction_state(friction_axes[0, 1], baseline_friction["final_low"], baseline_friction["final_high"], "")
friction_axes[0, 0].set_title("After settling; before velocity injection")
friction_axes[0, 1].set_title("After the measurement interval")
time = baseline_friction["time"]
for suffix, color, label in (("low", "#F27A24", "low μ"), ("high", "#2674C8", "high μ")):
    displacement = baseline_friction[f"x_{suffix}"] - baseline_friction[f"x_{suffix}"][0]
    friction_axes[1, 0].plot(time, displacement, color=color, label=f"{label} x")
    friction_axes[1, 0].plot(time, baseline_friction[f"vx_{suffix}"], linestyle="--", color=color, label=f"{label} vx")
    friction_axes[1, 1].plot(time, baseline_friction[f"omega_y_{suffix}"], color=color, label=label)
friction_axes[1, 0].set(xlabel="time [s]", ylabel="x [m] or vx [m/s]", title="Displacement and linear velocity")
friction_axes[1, 1].set(xlabel="time [s]", ylabel="angular velocity y [rad/s]", title="Rotation during sliding")
for axis in friction_axes[1]:
    axis.axhline(0.0, color="#444444", linewidth=1)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
friction_visual_source = "Genesis camera frames" if render_enabled else "Measured-state schematics — rendering disabled"
friction_figure.suptitle(f"{friction_visual_source}, with quantitative trajectories")
friction_figure.tight_layout()
friction_baseline_path = output_dir / "friction_baseline.png"
friction_figure.savefig(friction_baseline_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(friction_figure)

baseline_stop_lines = []
for label, suffix in (("Low-μ cube", "low"), ("High-μ cube", "high")):
    measurement = baseline_stop_measurements[suffix]
    if np.isfinite(measurement["time"]) and np.isfinite(measurement["distance"]):
        baseline_stop_lines.append(
            f"- {label} met the sustained-stop rule at {measurement['time']:.3f} s "
            f"after traveling {measurement['distance']:.4f} m."
        )
    else:
        baseline_stop_lines.append(
            f"- {label} did not meet the sustained-stop rule inside the measurement window."
        )

baseline_distances_finite = bool(np.isfinite([
    baseline_stop_measurements["low"]["distance"],
    baseline_stop_measurements["high"]["distance"],
]).all())
if not baseline_distances_finite:
    baseline_distance_verdict = "The stopping-distance ordering cannot be evaluated yet."
elif baseline_stop_measurements["low"]["distance"] > baseline_stop_measurements["high"]["distance"]:
    baseline_distance_verdict = "The lower-effective-friction lane traveled farther before sustained stopping."
elif baseline_stop_measurements["low"]["distance"] < baseline_stop_measurements["high"]["distance"]:
    baseline_distance_verdict = "The higher-effective-friction lane traveled farther; inspect the trajectories before accepting the prediction."
else:
    baseline_distance_verdict = "The two measured stopping distances were equal at the recorded precision."

max_abs_omega_low = float(np.max(np.abs(baseline_friction["omega_y_low"])))
max_abs_omega_high = float(np.max(np.abs(baseline_friction["omega_y_high"])))
contact_fraction_low = float(np.mean(baseline_friction["contacts_low"] > 0))
contact_fraction_high = float(np.mean(baseline_friction["contacts_high"] > 0))
friction_guidance = (
    "### Guided friction interpretation\n\n"
    "**1. Read the controlled comparison.**\n\n"
    + f"Both cubes use the same geometry, density, initial pose, table, outer dt "
    + f"({1000 * scalar(baseline_friction, 'dt'):.1f} ms), substeps "
    + f"({int(scalar(baseline_friction, 'substeps'))}), and initial vx "
    + f"({scalar(baseline_friction, 'initial_vx'):.3f} m/s). Only cube friction differs. "
    + f"With table μ={scalar(baseline_friction, 'table_friction'):.2f}, the cube values "
    + f"{scalar(baseline_friction, 'low_friction'):.2f} and "
    + f"{scalar(baseline_friction, 'high_friction'):.2f} produce effective pair values "
    + f"{scalar(baseline_friction, 'effective_low_friction'):.2f} and "
    + f"{scalar(baseline_friction, 'effective_high_friction'):.2f} under the locked "
    + "Genesis pair rule.\n\n"
    + "**2. Interpret sustained stopping, not one low-speed sample.**\n\n"
    + f"Stopping requires |vx| below {scalar(baseline_friction, 'stop_speed'):.3f} m/s "
    + f"for {scalar(baseline_friction, 'stop_hold'):.2f} s "
    + f"({int(scalar(baseline_friction, 'hold_samples'))} consecutive samples).\n\n"
    + "\n".join(baseline_stop_lines)
    + f"\n\n{baseline_distance_verdict} Read that ordering together with the full vx traces.\n\n"
    + "**3. Keep rotation and contact in the evidence chain.**\n\n"
    + f"Maximum absolute omega_y is {max_abs_omega_low:.4f} rad/s for the low-μ lane "
    + f"and {max_abs_omega_high:.4f} rad/s for the high-μ lane. Reported table contact "
    + f"is present in {100 * contact_fraction_low:.1f}% and "
    + f"{100 * contact_fraction_high:.1f}% of measured samples. A free cube can exchange "
    + "translation and rotation, so stopping distance alone is not the complete motion record.\n\n"
    + "**4. Bound the conclusion.**\n\n"
    + f"These observations come from Genesis {scalar(baseline_friction, 'genesis_version')} "
    + f"on the {scalar(baseline_friction, 'actual_backend')} backend, in this geometry and "
    + "measurement window. They test the cube-side material contrast while the table is "
    + "fixed. Surface colors only label the lanes and cannot explain the trajectory difference."
)
display(Markdown(friction_guidance))
print("saved:", friction_baseline_path.resolve())

## 单变量练习 · 只改变桌面，不改变方块

下一次运行只把桌面摩擦从 0.50 改为 0.30，其他输入全部不变。执行前先预测：哪个接触对有效摩擦系数会变化，哪条通道会移动得更远，哪条通道应近似不变。比较只检查变化方向和事先声明的容差，不要求复现某个硬编码停止距离。启用渲染时，基准与修改后的最终相机画面会和速度曲线一起提供场景层面的对照。

In [ ]:
MODIFIED_TABLE_FRICTION = 0.30
modified_spec = dict(FRICTION_SPEC)
modified_spec["table_friction"] = MODIFIED_TABLE_FRICTION
controlled_fields = set(FRICTION_SPEC) - {"table_friction"}
assert all(modified_spec[name] == FRICTION_SPEC[name] for name in controlled_fields)
modified_friction = run_friction_case(modified_spec, "table_friction_030.npz")
missing_modified_fields = FRICTION_FIELDS - set(modified_friction)
if missing_modified_fields:
    raise KeyError(f"modified friction result is missing fields: {sorted(missing_modified_fields)}")
validate_render_frames(modified_friction, "modified-table friction case")
assert scalar(modified_friction, "requested_backend") == backend_mode
for name, expected in modified_spec.items():
    assert np.isclose(scalar(modified_friction, name), expected), name

for name in FRICTION_ARRAY_FIELDS:
    values = modified_friction[name]
    assert values.shape == (expected_measure_samples,), (name, values.shape)
    assert np.isfinite(values).all(), name
modified_stop_measurements = {
    suffix: stop_measurement(modified_friction, suffix) for suffix in ("low", "high")
}
for suffix, measurement in modified_stop_measurements.items():
    for field in ("time", "distance"):
        saved = scalar(modified_friction, f"stop_{field}_{suffix}")
        recomputed = measurement[field]
        if np.isnan(recomputed):
            assert np.isnan(saved), (suffix, field)
        else:
            assert np.isclose(saved, recomputed), (suffix, field, saved, recomputed)
baseline_low_distance = baseline_stop_measurements["low"]["distance"]
baseline_high_distance = baseline_stop_measurements["high"]["distance"]
modified_low_distance = modified_stop_measurements["low"]["distance"]
modified_high_distance = modified_stop_measurements["high"]["distance"]
HIGH_LANE_UNCHANGED_TOLERANCE = 1e-3
distances_finite = bool(np.isfinite([
    baseline_low_distance, baseline_high_distance,
    modified_low_distance, modified_high_distance,
]).all())
if distances_finite:
    friction_direction_checks = {
        "baseline_low_lane_travels_farther": baseline_low_distance > baseline_high_distance,
        "modified_low_lane_travels_farther": modified_low_distance > baseline_low_distance,
        "high_lane_approximately_unchanged": (
            abs(modified_high_distance - baseline_high_distance) <= HIGH_LANE_UNCHANGED_TOLERANCE
        ),
    }
else:
    friction_direction_checks = {
        "baseline_low_lane_travels_farther": False,
        "modified_low_lane_travels_farther": False,
        "high_lane_approximately_unchanged": False,
    }
part_b_checks = {
    "all_stop_distances_finite": distances_finite,
    **friction_direction_checks,
    "low_effective_pair_changed": np.isclose(scalar(modified_friction, "effective_low_friction"), 0.30),
    "high_effective_pair_unchanged": np.isclose(scalar(modified_friction, "effective_high_friction"), 0.80),
}

comparison_lines = [
    "| Lane | baseline effective μ | modified effective μ | baseline stop [m] | modified stop [m] | change [m] |",
    "|---|---:|---:|---:|---:|---:|",
]
for label, suffix in (("Low-μ cube", "low"), ("High-μ cube", "high")):
    baseline_distance = baseline_stop_measurements[suffix]["distance"]
    modified_distance = modified_stop_measurements[suffix]["distance"]
    comparison_lines.append(
        f"| {label} | {scalar(baseline_friction, f'effective_{suffix}_friction'):.2f} | "
        f"{scalar(modified_friction, f'effective_{suffix}_friction'):.2f} | "
        f"{format_measurement(baseline_distance)} | {format_measurement(modified_distance)} | "
        f"{format_measurement(modified_distance - baseline_distance)} |"
    )
display(Markdown("\n".join(comparison_lines)))

exercise_scene_figure, exercise_scene_axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, result, title in (
    (exercise_scene_axes[0], baseline_friction, "Baseline table μ=0.50 — final"),
    (exercise_scene_axes[1], modified_friction, "Modified table μ=0.30 — final"),
):
    if render_enabled:
        axis.imshow(result["final_rgb"])
        axis.axis("off")
    else:
        draw_friction_state(axis, result["final_low"], result["final_high"], "")
    axis.set_title(title)
exercise_visual_source = "Genesis camera frames" if render_enabled else "Measured-state schematics — rendering disabled"
exercise_scene_figure.suptitle(f"{exercise_visual_source}: one-factor final-state comparison")
exercise_scene_figure.tight_layout()
friction_exercise_scene_path = output_dir / "friction_table_final_states.png"
exercise_scene_figure.savefig(friction_exercise_scene_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(exercise_scene_figure)

comparison_figure, comparison_axis = plt.subplots(figsize=(10, 4.5))
for result, style, table_label in (
    (baseline_friction, "-", "table μ=0.50"),
    (modified_friction, "--", "table μ=0.30"),
):
    comparison_axis.plot(result["time"], result["vx_low"], style, color="#F27A24", label=f"low-μ cube, {table_label}")
    comparison_axis.plot(result["time"], result["vx_high"], style, color="#2674C8", label=f"high-μ cube, {table_label}")
comparison_axis.axhline(0.0, color="#444444", linewidth=1)
comparison_axis.set(xlabel="time [s]", ylabel="vx [m/s]", title="Only table friction changes")
comparison_axis.grid(alpha=0.25)
comparison_axis.legend(fontsize=8)
comparison_figure.tight_layout()
friction_exercise_path = output_dir / "friction_table_comparison.png"
comparison_figure.savefig(friction_exercise_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(comparison_figure)

low_pair_before = scalar(baseline_friction, "effective_low_friction")
low_pair_after = scalar(modified_friction, "effective_low_friction")
high_pair_before = scalar(baseline_friction, "effective_high_friction")
high_pair_after = scalar(modified_friction, "effective_high_friction")
if not distances_finite:
    low_lane_result = "At least one low-lane stopping distance is unavailable, so its direction cannot be concluded."
    high_lane_result = "At least one high-lane stopping distance is unavailable, so the control comparison cannot be concluded."
else:
    low_change = modified_low_distance - baseline_low_distance
    high_change = modified_high_distance - baseline_high_distance
    low_lane_result = (
        f"The low lane traveled {low_change:.4f} m farther after the table change."
        if part_b_checks["modified_low_lane_travels_farther"]
        else f"The low-lane change was {low_change:+.4f} m and did not pass the expected direction check."
    )
    high_lane_result = (
        f"The high-lane change was {high_change:+.6f} m, within the declared "
        f"±{HIGH_LANE_UNCHANGED_TOLERANCE:.4f} m tolerance."
        if part_b_checks["high_lane_approximately_unchanged"]
        else f"The high-lane change was {high_change:+.6f} m, outside the declared "
        f"±{HIGH_LANE_UNCHANGED_TOLERANCE:.4f} m tolerance."
    )

one_factor_guidance = (
    "### Guided one-factor interpretation\n\n"
    "**1. Verify the intervention.**\n\n"
    + f"Only table friction changes from {scalar(baseline_friction, 'table_friction'):.2f} "
    + f"to {scalar(modified_friction, 'table_friction'):.2f}; the notebook has asserted "
    + f"that all {len(controlled_fields)} other command inputs are identical.\n\n"
    + "**2. Follow the effective pair values.**\n\n"
    + f"The low lane changes from μ={low_pair_before:.2f} to {low_pair_after:.2f} "
    + f"({'as predicted' if part_b_checks['low_effective_pair_changed'] else 'not as predicted'}), "
    + f"while the high lane changes from μ={high_pair_before:.2f} to {high_pair_after:.2f} "
    + f"({'unchanged as predicted' if part_b_checks['high_effective_pair_unchanged'] else 'unexpectedly changed'}).\n\n"
    + "**3. Compare measured stopping distances.**\n\n"
    + f"Low lane: {format_distance(baseline_low_distance)} → "
    + f"{format_distance(modified_low_distance)}. {low_lane_result}\n\n"
    + f"High lane: {format_distance(baseline_high_distance)} → "
    + f"{format_distance(modified_high_distance)}. {high_lane_result}\n\n"
    + "**4. State only what this one-factor experiment supports.**\n\n"
    + "A changed low lane together with an approximately unchanged high control lane supports "
    + "the table-side friction prediction for this Genesis version, scene, backend, and "
    + "observation window. It does not establish a universal friction law or guarantee the "
    + "same stopping distance on another solver, geometry, or timestep."
)
display(Markdown(one_factor_guidance))
print(part_b_checks)
print("saved:", friction_exercise_scene_path.resolve())
print("saved:", friction_exercise_path.resolve())

## 进入 L04 前的检查点

请根据刚刚生成的表格和图，用自己的话回答：

- 为什么 N1 和 N4 的内部时间步长相同，外层样本数却不同？
- 哪些信号组合能够支持或削弱“接触稳定”的判断？
- 为什么 `contact_count == 0` 不能单独证明方块发生反弹？
- Genesis 1.3.3 的接触对规则如何解释桌面摩擦练习中两条通道的不对称变化？
- 哪些结论只适用于当前场景、引擎版本、后端和观测窗口？

In [ ]:
backend_evidence = {
    scalar(case, "actual_backend")
    for case in (*contact_results.values(), baseline_friction, modified_friction)
}
backend_checks = {
    "one_actual_backend_used": len(backend_evidence) == 1,
    "forced_cpu_honored": backend_mode != "cpu" or backend_evidence == {"cpu"},
}
all_checks = {**part_a_checks, **part_b_checks, **backend_checks}
failed_checks = [name for name, passed in all_checks.items() if not passed]
for name, passed in all_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
print("requested backend:", backend_mode)
print("actual backend:", sorted(backend_evidence))
print("rendering:", "PASSED — Genesis camera frames captured" if render_enabled else "SKIP — disabled before build")
print("evidence directory:", output_dir.resolve())
if failed_checks:
    raise AssertionError("L03 checks failed: " + ", ".join(failed_checks))
print("L03 CHECK: PASSED")